# 14. 열량계 이상치 탐지 (Tdiff, Trl, Tvl, qv, V)

## 이상치 기준

### Tdiff (온도차, mK)
- 물리적 기준: 냉각 계량기 Tdiff > 0 (냉각은 공급온도 < 환수온도이므로 Tdiff 음수가 정상)
- 물리적 기준: 난방 계량기 Tdiff < 0 (난방은 공급온도 > 환수온도이므로 Tdiff 양수가 정상)
- 통계적 기준: 계량기별 일별 평균값 기준 평균 ± 3σ 초과

### Trl, Tvl (온도, °C)
- 물리적 기준: -20°C 미만 또는 120°C 초과 (배관 온도 물리적 한계)
- 통계적 기준: 평균 ± 3σ

### qv (유량, m³/h)
- 물리적 기준: qv < 0 (유량 음수 불가)
- 통계적 기준: 평균 ± 3σ

### V (누적유량)
- 물리적 기준: V < 0 (누적유량 음수 불가)
- 통계적 기준: 평균 ± 3σ

## 대상 계량기
냉각 열량계: V.K21, H1.K11, H1.K12, H1.K14, H1.K15, H1.K16, H2.K21
난방 열량계: H1.W11, H1.W12

In [1]:
import sys
from pathlib import Path

ROOT = Path('/home/aceya/EMS')
sys.path.insert(0, str(ROOT / 'src'))

import pandas as pd
import numpy as np
from ems.db import load_env, connect

load_env()

START = '2018-01-01'
END   = '2024-01-01'

COOLING = {'V.K21', 'H1.K11', 'H1.K12', 'H1.K14', 'H1.K15', 'H1.K16', 'H2.K21'}
HEATING = {'H1.W11', 'H1.W12'}
ALL_METERS = list(COOLING | HEATING)

save_dir = ROOT / 'outputs/tables/anomaly'
save_dir.mkdir(parents=True, exist_ok=True)
print('설정 완료')

설정 완료


In [2]:
def fetch_daily(meter_urn, measurement):
    sql = """
        SELECT
            DATE(ts AT TIME ZONE 'Europe/Berlin') AS day,
            MIN(value) AS min_val,
            MAX(value) AS max_val,
            AVG(value) AS avg_val,
            COUNT(*) AS cnt
        FROM ems.cr_measurement_1h
        WHERE meter_urn = %s
          AND measurement = %s
          AND ts >= %s
          AND ts <  %s
        GROUP BY 1
        ORDER BY 1
    """
    with connect() as conn:
        df = pd.read_sql(sql, conn, params=(meter_urn, measurement, START, END))
    if df.empty:
        return pd.DataFrame()
    df['day'] = pd.to_datetime(df['day'])
    return df


def detect_anomaly(meter, measurement):
    df = fetch_daily(meter, measurement)
    if df.empty:
        return None

    results = []

    # 물리적 기준 적용
    if measurement == 'Tdiff':
        if meter in COOLING:
            # 냉각: Tdiff > 0 이상
            physical = df[df['max_val'] > 0].copy()
            physical['anomaly_type'] = '물리적이상(냉각Tdiff양수)'
            physical['criterion'] = 'max_val > 0 mK (냉각 계량기에서 Tdiff 양수 불가)'
        else:
            # 난방: Tdiff < 0 이상
            physical = df[df['min_val'] < 0].copy()
            physical['anomaly_type'] = '물리적이상(난방Tdiff음수)'
            physical['criterion'] = 'min_val < 0 mK (난방 계량기에서 Tdiff 음수 불가)'
        if len(physical) > 0:
            results.append(physical)

    elif measurement in ['Trl', 'Tvl']:
        physical = df[(df['min_val'] < -20) | (df['max_val'] > 120)].copy()
        physical['anomaly_type'] = '물리적이상(온도범위초과)'
        physical['criterion'] = 'min_val < -20°C 또는 max_val > 120°C'
        if len(physical) > 0:
            results.append(physical)

    elif measurement == 'qv':
        physical = df[df['min_val'] < 0].copy()
        physical['anomaly_type'] = '물리적이상(음수유량)'
        physical['criterion'] = 'min_val < 0 m³/h'
        if len(physical) > 0:
            results.append(physical)

    elif measurement == 'V':
        physical = df[df['min_val'] < 0].copy()
        physical['anomaly_type'] = '물리적이상(음수누적유량)'
        physical['criterion'] = 'min_val < 0'
        if len(physical) > 0:
            results.append(physical)

    # 통계적 기준
    mean_val = df['avg_val'].mean()
    std_val  = df['avg_val'].std()
    if std_val > 0:
        upper = mean_val + 3 * std_val
        lower = mean_val - 3 * std_val
        stat = df[(df['avg_val'] > upper) | (df['avg_val'] < lower)].copy()
        stat['anomaly_type'] = '통계적이상(3sigma)'
        stat['criterion'] = f'mean={mean_val:.2f}, sigma={std_val:.2f}, lower={lower:.2f}, upper={upper:.2f}'
        if len(stat) > 0:
            results.append(stat)

    if not results:
        return None

    result = pd.concat(results).drop_duplicates('day').sort_values('day')
    result['meter'] = meter
    result['measurement'] = measurement
    return result[['meter', 'measurement', 'day', 'min_val', 'max_val', 'avg_val', 'anomaly_type', 'criterion']]

In [3]:
all_results = []

for meter in ALL_METERS:
    for meas in ['Tdiff', 'Trl', 'Tvl', 'qv', 'V']:
        result = detect_anomaly(meter, meas)
        if result is not None and len(result) > 0:
            print(f'{meter} {meas}: {len(result)}건')
            all_results.append(result)

if all_results:
    final = pd.concat(all_results, ignore_index=True)
    final.to_csv(save_dir / 'anomaly_thermal.csv', index=False)
    print(f'\n총 {len(final)}건 저장 완료')
else:
    print('이상치 없음')

/tmp/ipykernel_113102/4043608074.py:18: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(sql, conn, params=(meter_urn, measurement, START, END))


H1.K15 Tdiff: 393건


/tmp/ipykernel_113102/4043608074.py:18: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(sql, conn, params=(meter_urn, measurement, START, END))


H1.K15 Trl: 5건


/tmp/ipykernel_113102/4043608074.py:18: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(sql, conn, params=(meter_urn, measurement, START, END))


...

H1.K16 Trl: 39건


/tmp/ipykernel_113102/4043608074.py:18: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(sql, conn, params=(meter_urn, measurement, START, END))


H1.K16 Tvl: 33건


/tmp/ipykernel_113102/4043608074.py:18: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(sql, conn, params=(meter_urn, measurement, START, END))
/tmp/ipykernel_113102/4043608074.py:18: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(sql, conn, params=(meter_urn, measurement, START, END))



총 5835건 저장 완료


In [4]:
if all_results:
    summary = final.groupby(['meter', 'measurement', 'anomaly_type']).agg(
        건수=('day', 'count'),
        시작일=('day', 'min'),
        종료일=('day', 'max'),
        min_val=('min_val', 'min'),
        max_val=('max_val', 'max'),
        criterion=('criterion', 'first')
    ).reset_index()
    summary.to_csv(save_dir / 'anomaly_thermal_summary.csv', index=False)
    print(summary.to_string())

     meter measurement      anomaly_type    건수        시작일        종료일       min_val       max_val                                                     criterion
0   H1.K11       Tdiff  물리적이상(냉각Tdiff양수)  1137 2018-01-03 2023-12-18 -9.366333e+03  10539.583333                         max_val > 0 mK (냉각 계량기에서 Tdiff 양수 불가)
1   H1.K11       Tdiff     통계적이상(3sigma)    11 2020-08-08 2023-02-19 -6.607167e+03  -2159.416667    mean=-765.52, sigma=1280.64, lower=-4607.43, upper=3076.40
2   H1.K11         Trl     통계적이상(3sigma)    41 2018-01-02 2023-03-26  1.787500e+01     36.000000              mean=10.10, sigma=3.84, lower=-1.43, upper=21.62
3   H1.K11         Tvl     통계적이상(3sigma)    36 2018-11-11 2023-03-26  1.533333e+01     37.000000               mean=9.34, sigma=3.91, lower=-2.41, upper=21.08
4   H1.K12       Tdiff  물리적이상(냉각Tdiff양수)  1053 2018-01-03 2023-12-31 -1.515255e+04  17580.357186                         max_val > 0 mK (냉각 계량기에서 Tdiff 양수 불가)
5   H1.K12       Tdiff     통계적이상(3sigma)    14